In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

charts = pd.read_csv('/content/drive/MyDrive/music dataset from kwon/charts.csv')
spotify = pd.read_csv('/content/drive/MyDrive/music dataset from kwon/data_w_spotify.csv')
features = pd.read_csv('/content/drive/MyDrive/music dataset from kwon/feature_corel_100.csv')
features_simple = pd.read_csv('/content/drive/MyDrive/music dataset from kwon/feature_corel.csv')

print(charts.shape)
print(spotify.shape)
print(features.shape)
print(features_simple.shape)

(330087, 7)
(22517, 7)
(12905, 26)
(12905, 10)


In [3]:
!pip install pytrends

In [4]:
from pytrends.request import TrendReq
import pandas as pd
import time

# Initialize pytrends
pytrends = TrendReq(hl='en-US', tz=360)

# Get top 500 artists by Billboard appearances
charts = pd.read_csv('/content/drive/MyDrive/music dataset from kwon/charts.csv')
billboard_counts = charts.groupby('artist')['song'].count().reset_index()
billboard_counts.columns = ['artist', 'chart_appearances']
top_500 = billboard_counts.nlargest(500, 'chart_appearances')['artist'].tolist()

print(f'Total artists: {len(top_500)}')
print(f'Top 5: {top_500[:5]}')
print(f'Bottom 5: {top_500[-5:]}')

Total artists: 500
Top 5: ['Taylor Swift', 'Elton John', 'Madonna', 'Drake', 'Kenny Chesney']
Bottom 5: ['Herb Alpert', 'Ray Charles and his Orchestra', 'Spice Girls', 'Stevie Nicks', 'Frankie Valli']


In [6]:
youtube_results = {}
web_results = {}
failed_artists = []

for i, artist in enumerate(top_500):
    print(f'[{i+1}/500] Pulling: {artist}')

    # YouTube search
    for attempt in range(3):
        try:
            pytrends.build_payload(
                [artist],
                cat=35,
                timeframe='2004-01-01 2021-11-06',
                geo='US',
                gprop='youtube'
            )
            df_yt = pytrends.interest_over_time()
            if not df_yt.empty:
                youtube_results[artist] = df_yt[artist]
            time.sleep(1.5)
            break
        except Exception as e:
            print(f'  YouTube attempt {attempt+1} failed: {e}')
            if attempt < 2:
                time.sleep(10)
            else:
                failed_artists.append(f'{artist}_youtube')

    # Web search
    for attempt in range(3):
        try:
            pytrends.build_payload(
                [artist],
                cat=35,
                timeframe='2004-01-01 2021-11-06',
                geo='US',
                gprop=''
            )
            df_web = pytrends.interest_over_time()
            if not df_web.empty:
                web_results[artist] = df_web[artist]
            time.sleep(1.5)
            break
        except Exception as e:
            print(f'  Web attempt {attempt+1} failed: {e}')
            if attempt < 2:
                time.sleep(10)
            else:
                failed_artists.append(f'{artist}_web')

    # Checkpoint every 50 artists
    if (i + 1) % 50 == 0:
        print(f'\n--- Checkpoint {i+1}/500 ---')
        temp_yt = pd.DataFrame(youtube_results)
        temp_web = pd.DataFrame(web_results)
        temp_yt.columns = [f'{c}_youtube' for c in temp_yt.columns]
        temp_web.columns = [f'{c}_web' for c in temp_web.columns]
        temp = pd.concat([temp_yt, temp_web], axis=1)
        temp = temp.reindex(sorted(temp.columns), axis=1)
        temp.to_csv(f'/content/drive/MyDrive/Googletrend dataset/trends_checkpoint_{i+1}.csv')
        print(f'Checkpoint saved! Failed so far: {len(failed_artists)}')

[1/500] Pulling: Taylor Swift
[2/500] Pulling: Elton John
[3/500] Pulling: Madonna
[4/500] Pulling: Drake
[5/500] Pulling: Kenny Chesney
[6/500] Pulling: Tim McGraw
[7/500] Pulling: Keith Urban
[8/500] Pulling: Stevie Wonder
[9/500] Pulling: Rod Stewart
[10/500] Pulling: Mariah Carey
[11/500] Pulling: Michael Jackson
[12/500] Pulling: Chicago
[13/500] Pulling: Rascal Flatts
[14/500] Pulling: Billy Joel
[15/500] Pulling: The Beatles
[16/500] Pulling: The Rolling Stones
[17/500] Pulling: Jason Aldean
[18/500] Pulling: Aretha Franklin
[19/500] Pulling: Rihanna
[20/500] Pulling: P!nk
[21/500] Pulling: Whitney Houston
[22/500] Pulling: Brad Paisley
[23/500] Pulling: George Strait
[24/500] Pulling: Neil Diamond
[25/500] Pulling: Luke Bryan
[26/500] Pulling: Carrie Underwood
[27/500] Pulling: Daryl Hall John Oates
[28/500] Pulling: The Beach Boys
[29/500] Pulling: Toby Keith
[30/500] Pulling: Bee Gees
[31/500] Pulling: Blake Shelton
[32/500] Pulling: Maroon 5
[33/500] Pulling: Kelly Clarkson


In [7]:
youtube_df = pd.DataFrame(youtube_results)
youtube_df.columns = [f'{col}_youtube' for col in youtube_df.columns]

web_df = pd.DataFrame(web_results)
web_df.columns = [f'{col}_web' for col in web_df.columns]

trends_final = pd.concat([youtube_df, web_df], axis=1)
trends_final = trends_final.reindex(sorted(trends_final.columns), axis=1)

print(f'Shape: {trends_final.shape}')
print(f'Date range: {trends_final.index.min()} to {trends_final.index.max()}')
print(f'Failed artists: {len(failed_artists)}')
print(f'Failed list: {failed_artists}')
print(trends_final.head())

Shape: (215, 993)
Date range: 2004-01-01 00:00:00 to 2021-11-01 00:00:00
Failed artists: 2
Failed list: ['Marty Robbins_web', 'Tavares_youtube']
            'N Sync_web  2Pac_web  2Pac_youtube  3 Doors Down_web  \
date                                                                
2004-01-01            0        97             0                95   
2004-02-01            0        86             0                68   
2004-03-01            0        75             0                61   
2004-04-01            0        74             0                54   
2004-05-01            0        73             0                40   

            3 Doors Down_youtube  50 Cent_web  50 Cent_youtube  ABBA_web  \
date                                                                       
2004-01-01                     0           20                0        37   
2004-02-01                     0           21                0        31   
2004-03-01                     0           17                0     

In [8]:
trends_final.to_csv(
    '/content/drive/MyDrive/Googletrend dataset/google_trends_top500.csv'
)
print('Saved!')

Saved!


In [9]:
print(trends_final.head())

            'N Sync_web  2Pac_web  2Pac_youtube  3 Doors Down_web  \
date                                                                
2004-01-01            0        97             0                95   
2004-02-01            0        86             0                68   
2004-03-01            0        75             0                61   
2004-04-01            0        74             0                54   
2004-05-01            0        73             0                40   

            3 Doors Down_youtube  50 Cent_web  50 Cent_youtube  ABBA_web  \
date                                                                       
2004-01-01                     0           20                0        37   
2004-02-01                     0           21                0        31   
2004-03-01                     0           17                0        32   
2004-04-01                     0           14                0        39   
2004-05-01                     0           14               